In [4]:
"""
导包
""" 


import os
from pathlib import Path
from sdp.dataset.base_dataset import BaseDataset
from csi.csi_data import CSIData
from sdp.reader.reader_factory import ReaderFactory
from sdp.dataset.dataset_factory import DatasetFactory
from sdp.processor.processor_factory import ProcessorFactory

In [5]:
"""
设置目录 创建主函数
"""


PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# 主函数
def process_file_from_folder(folder_path: str) -> BaseDataset:
    folder_path = Path(folder_path)
    if not folder_path.exists() or not folder_path.is_dir():
        raise ValueError(f"无效的文件夹路径: {folder_path}")
    files = [f for f in folder_path.rglob("*") if f.is_file() and "truth" not in f.name]
    if not files:
        print(f"文件夹 {folder_path} 中没有文件")
        return None
    
    # 使用第一个文件确定主格式
    sample_file = files[0]
    reader = ReaderFactory.create_reader(str(sample_file))
    sample_frame = reader.read_file(sample_file).frames[0]
    processor = ProcessorFactory.get_processor(sample_frame)
    

    print(f"检测到主文件格式: {type(reader).__name__}")
    
    print(f"开始处理 {len(files)} 个文件...")
    
    csi_data_list = []
    # 处理所有文件
    for file_path in files:
        try:
            csi_data = reader.read_file(str(file_path))
            csi_data_list.append(csi_data)
            
            print(f"√ 已处理: {file_path.name}")
        
        except Exception as e:
            print(f"× 处理失败 {file_path.name}: {str(e)}")
    
    print(f"处理完成! 共处理 {len(files)} 个文件")
    
    # 处理csi_data_list 转换成对应参数
    res = list(processor.process(csi_data_list, folder_path))
    # 解包元组 构造dataset
    # dataset = DatasetFactory.create_dataset(res, reader)
    
    return dataset

In [6]:
from pprint import pprint

"""
执行位置
"""


folder_path = PROJECT_ROOT / "data/hw_data"
dataset = process_file_from_folder(folder_path)
#pprint(f"the final dataset: {vars(dataset)}")

检测到主文件格式: HwOfficeReader
开始处理 1 个文件...
√ 已处理: csi_1008_2023_10_30_1.txt
处理完成! 共处理 1 个文件
6686

随机选取的数据来自文件: csi_1008_2023_10_30_1.txt

=== 第 1 条 CSI 数据包 ===
Timestamp (ts): [0.0, 0.0, 0.0]
RSSI: [-67.0, -71.0]
MCS: 22.0
Gain: [142.0, 206.0]
CSI (full data):

=== 第 2 条 CSI 数据包 ===
Timestamp (ts): [0.0, 0.0, 0.0031]
RSSI: [-66.0, -70.0]
MCS: 22.0
Gain: [142.0, 190.0]
CSI (full data):

=== 第 最后 条 CSI 数据包 ===
Timestamp (ts): [0.0, 4.0, 59.0996]
RSSI: [-70.0, -71.0]
MCS: 23.0
Gain: [206.0, 222.0]
CSI (full data):
[load_data_from_folder] 读取 1 个文件于 F:\Repository\pyProjects\sdp_benchmark\data\hw_data

处理文件 csi_1008_2023_10_30_1.txt: 
truth_path:F:\Repository\pyProjects\sdp_benchmark\data\hw_data\home_scenario_1\data\room_A\csi_1008_2023_10_30_1_truth.txt
  对应 truth 数目: 150
  目标采样率: 20 Hz, 均匀时间长度: 5982, num_rssi =2
  插值后 csi: (5982, 30, 4) rssi: (5982, 2)
  scaled_tensor shape=(5982, 30, 4)
[clamp] highcut=40.0超出Nyquist=10.0,自动缩小
[Info] 采样率 fs=20Hz, lowcut=2.0Hz, highcut=9.9Hz
nperseg: 40
nfft: 